<a href="https://colab.research.google.com/github/Sirukk/data-engineering-2026/blob/main/LR01_setup_v2_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота 1. Налаштування робочого середовища

**Дисципліна:** Інженерія даних з Python  •  **Обсяг:** 2 год  •  **Тема 1**  •  **Версія 2**

Кафедра комп'ютерних наук, Кам'янець-Подільський національний університет імені Івана Огієнка

---

**Виконала:**

Студент(ка) групи: **KN1-B23**

Прізвище, ім'я: **Сірук Аліна**

Посилання на блокнот:___________________
---

**Домашня підготовка:** мікрокурс [Python](https://www.kaggle.com/learn/python) платформи Kaggle Learn, завдання 1–7.

**Опорний конспект теми:** `T01_intro_pandas.ipynb`

> **Перш ніж почати:** оберіть *Файл → Зберегти копію на Диску*.

## Мета роботи

Налаштувати середовище, у якому виконуватимуться всі наступні лабораторні роботи семестру, і переконатися, що всі канали доступу до даних працюють.

**Після виконання ви матимете:**

1. робочу теку на Google Диску;
2. налаштований доступ до Kaggle через програмний інтерфейс, з ключем у секретах Colab;
3. налаштований доступ до Google BigQuery через режим Sandbox;
4. власний репозиторій GitHub із збереженим блокнотом.

**Найдовший крок** — створення проєкту в Google Cloud. Зробіть його вдома напередодні заняття, тоді на парі залишиться саме робота, а не очікування.

> **Важливо:** усю роботу виконуйте під **власним** обліковим записом Google. Секрети Colab прив'язані до акаунта, тому під спільним акаунтом лабораторії налаштування не збережеться.

## Крок 0. Перевірка середовища

Перша комірка кожного блокнота курсу друкує версії бібліотек. Це не формальність: Colab оновлюється без попередження, і якщо код колись перестане працювати, ви одразу побачите, що змінилося.

In [ ]:
import sys
import numpy as np
import pandas as pd

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy :", np.__version__)

pd.set_option("display.max_rows", 12)
pd.set_option("display.width", 120)

## Крок 1. Google Диск

Обчислювальна сесія Colab тимчасова: після її зупинки всі створені файли зникають. Тому результати роботи зберігаємо на Диск.

Виконайте комірку — Colab запитає дозвіл.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path

РОБОЧА = Path('/content/drive/MyDrive/data-engineering/lab01')
РОБОЧА.mkdir(parents=True, exist_ok=True)

print("Робоча тека:", РОБОЧА)
print("Існує:", РОБОЧА.exists())

Робоча тека: /content/drive/MyDrive/data-engineering/lab01
Існує: True


### Завдання 1.1

Запишіть у робочу теку файл і прочитайте його назад. Це найпростіша перевірка того, що Диск підключений на запис, а не лише на читання.

Створіть таблицю з **трьох рядків**, у якій будуть ваше прізвище, номер за списком групи і поточна дата. Збережіть її у файл `РОБОЧА / 'perevirka.csv'` і прочитайте назад у змінну `perevirka`.

> Не забудьте `index=False` при записі — інакше при читанні зʼявиться зайвий стовпець `Unnamed: 0`.

In [ ]:
from datetime import date

# ВАШ КОД ТУТ
# 1. Створіть DataFrame з трьох рядків і принаймні двох стовпців
# 2. Збережіть його у РОБОЧА / 'perevirka.csv' з index=False
# 3. Прочитайте назад у perevirka
import pandas as pd
from pathlib import Path

РОБОЧА = Path('.')

perevirka = pd.DataFrame({
    'Параметр': ['Прізвище', 'Номер за списком', 'Поточна дата'],
    'Значення': ['Сірук', 26, date.today()]
})

perevirka.to_csv(РОБОЧА / 'perevirka.csv', index=False)

perevirka = pd.read_csv(РОБОЧА / 'perevirka.csv')

print(perevirka)

           Параметр    Значення
0          Прізвище       Сірук
1  Номер за списком          26
2      Поточна дата  2026-09-16


## Крок 2. Kaggle API

Програмний інтерфейс Kaggle дозволяє завантажувати набори даних командою, а не через браузер. Це важливо для відтворюваності: команда в блокноті документує, звідки взялися дані, а завантаження мишею — ні.

Доступ до API потребує двох значень — імені користувача й ключа. Зберігати їх ми будемо в **секретах Colab**, а не в коді блокнота.

### 2.1. Отримання облікових даних

1. Відкрийте kaggle.com → ваш профіль → *Settings* → вкладка *API Tokens*.
2. На сторінці **дві секції**. Потрібна **нижня** — **Legacy API Credentials**. Натисніть у ній кнопку **`Create Legacy API Key`**.

> **Верхня секція не підходить.** Вона називається *API Tokens (Recommended)* і має кнопку `Generate New Token`. Попри слово «Recommended», цей новий формат наш код не приймає: він працює з парою «ім'я користувача + ключ», а новий токен — це одне значення іншого вигляду. Тиснемо саме `Create Legacy API Key`.

3. Завантажиться файл `kaggle.json`. Окремої кнопки «показати токен» на Kaggle немає — значення лежать усередині цього файлу.
4. Відкрийте `kaggle.json` **будь-яким текстовим редактором**. Усередині два поля:

```
{"username":"ivan_student","key":"3f8a2b9c1d4e..."}
```

5. Ці два значення знадобляться на наступному кроці. Нікуди їх не пересилайте — вони підуть одразу в секрети Colab.

### 2.2. Збереження в секретах Colab

Секрет складається з двох частин: **назви** і **значення**. Назва — це ярлик, за яким код знаходить секрет; вона однакова в усіх і **вже прописана в коді нижче**. Значення — ваше особисте, з файлу `kaggle.json`.

| Поле «Назва» | Поле «Значення» |
|---|---|
| `KAGGLE_USERNAME` | ваш нікнейм — те, що у файлі стоїть після `"username":` |
| `KAGGLE_KEY` | довгий ключ — те, що у файлі стоїть після `"key":` |

1. На лівій панелі Colab натисніть іконку **ключа** (*Secrets* / *Секрети*), далі **«+ Додати секрет»**.
2. Заповніть перший рядок за таблицею вище, потім додайте другий секрет і заповніть його.
3. Для обох увімкніть перемикач **«Доступ для блокнота»** — він зліва від рядка і за замовчуванням **вимкнений**.

> **Код у комірці нижче змінювати не потрібно.** Рядки `userdata.get('KAGGLE_USERNAME')` і `userdata.get('KAGGLE_KEY')` уже правильні: у дужках стоїть **назва** секрету, за якою Colab піде в панель і забере значення. Якщо вписати туди свій нікнейм або ключ, отримаєте `SecretNotFoundError: Secret <ваш нікнейм> does not exist` — Colab шукатиме секрет із такою назвою і не знайде його.

> **Чому саме так, а не змінними в комірці.** Секрети прив'язані до вашого Google-акаунта, а не до сесії: записали один раз — працює до кінця обох семестрів. Головне ж у тому, що ключ не потрапляє у файл блокнота. Якби ви вписали його прямо в код, він поїхав би разом із `.ipynb` у Moodle і в репозиторій GitHub — а GitHub боти сканують на такі ключі цілодобово.

> Це ваш особистий ключ. Не публікуйте його і не надсилайте нікому — ні викладачу, ні в чат групи. Якщо ключ усе ж кудись потрапив, це не катастрофа: поверніться в *Settings* → *API Tokens*, натисніть **`Expire Legacy API Key`**, потім **`Create Legacy API Key`** і впишіть у секрет нове значення.

In [1]:
!pip install -q kaggle

In [4]:
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')

print("Облікові дані підставлено для користувача:", os.environ['KAGGLE_USERNAME'])

Облікові дані підставлено для користувача: sirukk6c7d9914f4f2a3d6098e183085a9eee2


In [5]:
# Перевірка звʼязку: команда має вивести список без помилки автентифікації
!kaggle datasets list -s "books" --max-size 52428800 --file-type csv

ref                                                               title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
----------------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
jealousleopard/goodreadsbooks                                     Goodreads-books                                         637338  2020-03-09 09:18:31.583000          92090       2021                1  
saurabhbagchi/books-dataset                                       Books Dataset                                         25760320  2020-10-09 05:14:41.297000          25828        146                1  
abdallahwagih/books-dataset                                       Books Dataset                                          1657228  2023-12-13 02:26:07.263000           7870        103          

### Завдання 2.3

Знайдіть на Kaggle набір даних, який вам цікавий, завантажте його і прочитайте у `pandas`.

Повторіть пошук за власним ключовим словом — замініть `books` на будь-яку тему, що вас цікавить.

> **Частина наборів не завантажиться, і це нормально.** Якщо у відповідь прийде `403 Client Error: Forbidden` — набір вимагає прийняти умови на його сторінці в браузері. Або прийміть їх, або просто візьміть наступний набір зі списку. Це не помилка у вашому коді.

In [6]:
# ВАШ КОД ТУТ
# Виконайте пошук за своїм ключовим словом
# !kaggle datasets list -s "ваше_слово" --file-type csv
!kaggle datasets list -s "movies" --max-size 52428800 --file-type csv

ref                                                             title                                            size  lastUpdated                 downloadCount  voteCount  usabilityRating  
--------------------------------------------------------------  -----------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
shivamb/netflix-shows                                           Netflix Movies and TV Shows                   1400865  2021-09-27 04:44:36.770000         806429      10133                1  
harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows  IMDB Movies Dataset                            179262  2021-02-01 07:35:48.597000         123698        797                1  
abdallahwagih/movies                                            Movies                                        5377152  2023-09-22 13:41:56.597000           3844         60                1  
asaniczka/tmdb-movies-dataset-2023-930k-movie

З виведеного списку **оберіть один набір розміром до 50 МБ**. Ідентифікатор — це перший стовпець у форматі `власник/назва-набору`.

> **Якщо `read_csv` упаде з `ParserError: Error tokenizing data`** — файл не зламаний, просто в ньому інший роздільник (часто крапка з комою замість коми). Спершу подивіться на сирі рядки файлу, а вже потім читайте з правильним `sep`:
>
> ```python
> рядки = open(шлях, encoding="utf-8", errors="replace").read().splitlines()[:6]
> for i, r in enumerate(рядки): print(i, "|", r[:110])
> ```
>
> Побачили `;` замість `,` — читайте `pd.read_csv(шлях, sep=";")`. Це ваша перша зустріч із «сирими» даними; докладно ними займемося в темі 6.

In [7]:
# ВАШ КОД ТУТ
# 1. Впишіть обраний ідентифікатор
# 2. Розкоментуйте команди завантаження

# !kaggle datasets download -d ВЛАСНИК/НАЗВА -p /content/kaggle_data --unzip
# !ls -lh /content/kaggle_data
!kaggle datasets download -d shivamb/netflix-shows -p /content/kaggle_data --unzip
!ls -lh /content/kaggle_data

Dataset URL: https://www.kaggle.com/datasets/shivamb/netflix-shows
License(s): CC0-1.0
100% 1.34M/1.34M [00:00<00:00, 2.52MB/s]

total 3.3M
-rw-r--r-- 1 root root 3.3M Sep 19 19:52 netflix_titles.csv


In [9]:
# ВАШ КОД ТУТ
# 3. Прочитайте один із завантажених CSV у DataFrame з іменем kaggle_df
# 4. Виведіть форму таблиці та перші рядки
import pandas as pd

шлях = "/content/kaggle_data/netflix_titles.csv"

kaggle_df = pd.read_csv(шлях)

print("Форма таблиці:", kaggle_df.shape)
kaggle_df.head()


Форма таблиці: (8807, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


## Крок 3. Google BigQuery Sandbox

BigQuery — хмарне аналітичне сховище, з яким ми працюватимемо в темах 4 і 5. Режим **Sandbox** дає безкоштовний доступ без прив'язки платіжної картки, з місячним лімітом обробки запитів у 1 ТБ. Для всіх завдань курсу цього вистачає з великим запасом.

**Підготовка.** Відкрийте console.cloud.google.com, створіть проєкт (назва довільна, наприклад `de-kpnu-2026`) і скопіюйте його **Project ID**.

> **Name і Project ID — різні речі.** Name — те, що ви вписали самі; Project ID — те, що згенерував Google, і воно може мати числовий хвіст (`de-kpnu-2026-472118`). Потрібен саме **ID**: він видно в колонці «ID» у селекторі проєктів або в адресному рядку після `project=`.

> **Зайдіть один раз у BigQuery через консоль** (меню ☰ → BigQuery). Перше відкриття активує Sandbox і вмикає BigQuery API для проєкту. Без цього з Colab прилетить `403 BigQuery API has not been used in project … or it is disabled`.

> **Якщо комірка автентифікації завершиться помилкою** `MessageError: Error: credential propagation was unsuccessful` — доступ ви надали, просто блокнот не встиг його підхопити. Виконайте цю саму комірку ще раз.

In [13]:
from google.colab import auth
auth.authenticate_user()
print("Автентифікацію виконано")

Автентифікацію виконано


In [14]:
from google.cloud import bigquery

PROJECT_ID = "de-kpnu-2026-509119"          # <<< впишіть ідентифікатор вашого проєкту
if PROJECT_ID.strip() in ("", "..."):
    raise ValueError(
        "Спершу впишіть ідентифікатор свого проєкту Google Cloud у змінну PROJECT_ID вище "
        "(console.cloud.google.com → оберіть проєкт → скопіюйте Project ID), "
        "після чого виконайте цю комірку ще раз."
    )

client = bigquery.Client(project=PROJECT_ID)

print("Клієнт створено для проєкту:", client.project)

Клієнт створено для проєкту: de-kpnu-2026-509119


### Завдання 3.1

Працюємо з публічним набором `bigquery-public-data.usa_names` — це імена новонароджених у США за роками.

Подивіться, які в наборі є таблиці, і оберіть одну з них.

> У комірці нижче вже стоїть робоче значення `usa_1910_2013` — блокнот виконується від початку до кінця без помилок. Замініть його на будь-яку іншу таблицю зі списку, якщо хочете.

In [16]:
dataset_ref = "bigquery-public-data.usa_names"
tables = list(client.list_tables(dataset_ref))

print(f"Набір містить {len(tables)} таблиць:")
for t in tables:
    print("  ", t.table_id)

Набір містить 2 таблиць:
   usa_1910_2013
   usa_1910_current


In [17]:
# ВАШ КОД ТУТ
# Оберіть одну таблицю зі списку вище і виведіть:
# кількість рядків, обсяг у мегабайтах, перелік стовпців.
#
# Підказка: info = client.get_table(f"{dataset_ref}.НАЗВА")
#           у info є num_rows, num_bytes, schema

# ВАШ КОД ТУТ
# Оберіть одну таблицю зі списку вище і виведіть:
# кількість рядків, обсяг у мегабайтах, перелік стовпців.

ТАБЛИЦЯ = "usa_1910_2013"

table_info = client.get_table(f"{dataset_ref}.{ТАБЛИЦЯ}")

print("Кількість рядків:", table_info.num_rows)

size_mb = table_info.num_bytes / (1024 * 1024)
print(f"Обсяг таблиці: {size_mb:.2f} МБ")

print("\nСтовпці:")
for field in table_info.schema:
    print(f"  {field.name} — {field.field_type}")

Кількість рядків: 5552452
Обсяг таблиці: 163.49 МБ

Стовпці:
  state — STRING
  gender — STRING
  year — INTEGER
  name — STRING
  number — INTEGER


### Завдання 3.2. Оцінка вартості запиту

У BigQuery ви платите за обсяг **просканованих** даних, а не за кількість повернутих рядків. Тому `LIMIT 10` не робить запит дешевим: якщо запит читає всю таблицю, ви заплатите за всю таблицю.

Оцінити обсяг можна заздалегідь, не виконуючи запит, — це **сухий прогін** (dry run). Звикайте робити його перед кожним новим запитом.

> **Якого числа очікувати.** Не десятикратної різниці. Економія приблизно дорівнює частці обраних стовпців у загальному обсязі таблиці: якщо ви взяли два «важких» стовпці з п'яти, різниця вийде близько двох разів — і це правильний результат, а не ознака помилки. Щоб побачити більший контраст, оцініть додатково запит до одного числового стовпця.

In [18]:
def оцінити(sql):
    """Скільки даних просканує запит, не виконуючи його."""
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    job = client.query(sql, job_config=cfg)
    mb = job.total_bytes_processed / 1024**2
    print(f"Просканує {mb:,.1f} МБ")
    return job.total_bytes_processed

if ТАБЛИЦЯ.strip() in ("", "..."):
    розмір_усі = None
    print("Спершу впишіть назву таблиці у змінну ТАБЛИЦЯ (завдання 3.1) і виконайте цю комірку ще раз.")
else:
    sql_all = f"SELECT * FROM `{dataset_ref}.{ТАБЛИЦЯ}` LIMIT 10"
    розмір_усі = оцінити(sql_all)

Просканує 163.5 МБ


In [19]:
# ВАШ КОД ТУТ
# Складіть другий запит: ті самі 10 рядків, але лише ДВА стовпці.
# Оцініть його сухим прогоном і покладіть результат у розмір_два.

sql_two = f"""
SELECT name, number
FROM `{dataset_ref}.{ТАБЛИЦЯ}`
LIMIT 10
"""

розмір_два = оцінити(sql_two)

Просканує 84.1 МБ


In [20]:
# ВАШ КОД ТУТ
# Виконайте другий запит по-справжньому, результат покладіть у bq_df.
# Підказка: client.query(sql).to_dataframe()

bq_df = client.query(sql_two).to_dataframe()
bq_df

,name,number
0,Sadie,40
1,Mary,875
2,Vera,39
3,Marie,78
4,Lucille,66
5,Virginia,101
6,Margaret,72
7,Mildred,133
8,Vera,51
9,Sallie,92


> **Запишіть у комірці нижче:** у скільки разів відрізнявся обсяг сканування між запитом з `SELECT *` і запитом із двома стовпцями. Це число знадобиться в темі 5.

**Ваша відповідь:** обсяг сканування зменшився у 1,94 рази, оскільки запит із двома стовпцями сканує менше даних, ніж SELECT *, який зчитує всі стовпці таблиці.

_(поясніть одним реченням, чому)_

## Крок 4. GitHub

Матеріали курсу зберігаються в репозиторії, а ваші роботи протягом семестру складатимуться у власний. Для фінального проєкту репозиторій обовʼязковий — це прямий наслідок вимоги відтворюваності.

**Створіть репозиторій** з іменем `data-engineering-2026` на github.com (приватний або публічний — на ваш розсуд).

### Завдання 4.1

Збережіть цей блокнот у свій репозиторій: *Файл → Зберегти копію на GitHub*.

> **Не пишіть у коміті «update».** Коментар має відповідати на питання «що змінилося»: наприклад, «ЛР 1: налаштування середовища».

---

## Висновки

*Напишіть 3–5 речень про те, що ви зробили в цій роботі та що з цього винесли.*

_(впишіть висновки тут)_